# Лекция 04. Применение Python

От отдельных алгоритмов переходим к задачам, приложениям и осознанному выбору проекта.

## Цели

После лекции вы сможете:

- различать основные области применения Python;
- видеть общую архитектуру прикладной программы;
- отделять предметное ядро от внешних интерфейсов;
- объяснить роль Python в анализе данных, ML и ИИ;
- назвать границы применимости языка;
- выбрать проект от проблемы, а не от технологии;
- сформулировать минимальную демонстрируемую версию проекта;
- объяснить, как формируется команда, регистрируется тема и проходит проектная работа до защиты.

## Перед началом

Это обзорная лекция: она показывает классы задач и типовую архитектуру, но не заменяет специальные занятия по объектной модели, HTTP, асинхронности, базам данных и поставке приложения.

Не нужно запоминать названия всех библиотек. Важнее научиться задавать вопросы: кто пользователь, какие данные входят, какое вычисление составляет ядро и как наблюдать результат.

## Только запускаемые примеры

Все основные примеры лежат в каталоге [`examples`](examples/) как обычные `.py`-файлы:

- [`csv_report.py`](examples/csv_report.py) — CLI-отчёт из настоящего CSV;
- [`linear_regression.py`](examples/linear_regression.py) — модель scikit-learn;
- [`fastapi_app.py`](examples/fastapi_app.py) — работающий HTTP endpoint;
- [`telegram_echo_bot.py`](examples/telegram_echo_bot.py) — настоящий echo-bot через Telegram Bot API;
- [`pygame_ball.py`](examples/pygame_ball.py) — окно, игровой цикл и движущийся объект.

Команды установки и запуска записаны в [`examples/README.md`](examples/README.md). Код намеренно минимален: фреймворк обычно добавляет тонкий слой вокруг уже знакомых функций, условий и циклов.

## Первое приложение: CSV → отчёт

Файл [`csv_report.py`](examples/csv_report.py) делает реальную работу: принимает путь из командной строки, читает CSV, суммирует продажи по категориям и печатает отчёт. Внутри нет новой магии — `argparse`, `csv`, цикл, преобразование `int` и `print`.

Запуск:

```bash
python lesson04/examples/csv_report.py lesson04/examples/sales.csv
```

Даже такое приложение уже имеет вход, предметное вычисление и выход. На семинаре разнесём эти части по модулям, чтобы их было легче проверять и менять.

In [ ]:
import csv
from pathlib import Path

sales_path = Path("lesson04/examples/sales.csv")
if not sales_path.exists():
    sales_path = Path("examples/sales.csv")

totals: dict[str, int] = {}
with sales_path.open(encoding="utf-8", newline="") as source:
    for row in csv.DictReader(source):
        category = row["category"]
        totals[category] = totals.get(category, 0) + int(row["amount"])

for category, total in totals.items():
    print(f"{category}: {total}")

## Автоматизация: переименовать файлы

Следующая ячейка действительно создаёт два файла и переименовывает их через `Path.rename`. Чтобы демонстрация не затронула ваши данные, используется временный каталог, который удаляется автоматически.

В реальном скрипте меняется только источник путей. Полезная привычка — сначала напечатать план переименования, проверить коллизии имён и лишь затем выполнять изменения.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    folder = Path(directory)
    for name in ["Report January.csv", "Report February.csv"]:
        (folder / name).write_text("amount\n100\n", encoding="utf-8")

    for old_path in folder.glob("*.csv"):
        new_name = old_path.name.lower().replace(" ", "-")
        old_path.rename(folder / new_name)

    print(sorted(path.name for path in folder.iterdir()))

## Анализ данных

Аналитическое приложение обычно проходит конвейер:

`получение → проверка → очистка → преобразование → агрегация → визуализация → вывод`.

Python популярен здесь благодаря выразительному языку и библиотекам для массивов, таблиц, статистики и графиков. Но библиотека не исправляет плохое определение показателя, пропуски или смещённую выборку. Аналитик отвечает и за код, и за смысл результата.

In [ ]:
from statistics import mean

with sales_path.open(encoding="utf-8", newline="") as source:
    amounts = [int(row["amount"]) for row in csv.DictReader(source)]

print(f"rows: {len(amounts)}")
print(f"total: {sum(amounts)}")
print(f"mean: {mean(amounts):.2f}")

## Машинное обучение: четыре рабочих действия

Файл [`linear_regression.py`](examples/linear_regression.py) импортирует `LinearRegression`, создаёт модель, вызывает `fit` на пяти наблюдениях и `predict` для следующей недели. Это уже настоящая модель scikit-learn, а не вручную написанная формула.

```bash
python lesson04/examples/linear_regression.py
```

Технически обучение действительно занимает несколько строк. Сложность ML-проекта находится не в вызове `fit`, а в определении цели, качестве данных, честном разделении выборки и интерпретации ошибки.

In [ ]:
try:
    from sklearn.linear_model import LinearRegression
except ModuleNotFoundError:
    print("Установите зависимости из lesson04/examples/requirements.txt")
else:
    weeks = [[1], [2], [3], [4], [5]]
    sales = [10, 12, 15, 17, 20]

    model = LinearRegression()
    model.fit(weeks, sales)
    prediction = model.predict([[6]])[0]
    print(f"week 6: {prediction:.1f}")

## FastAPI: обычная функция плюс маршрут

В [`fastapi_app.py`](examples/fastapi_app.py) всего одна функция. Декоратор `@app.get("/hello")` связывает её с HTTP GET-запросом, параметр `name: str` берётся из query string, а возвращаемый словарь превращается в JSON.

```bash
cd lesson04/examples
uvicorn fastapi_app:app --reload
```

После запуска откройте `http://127.0.0.1:8000/hello?name=Student` и `/docs`. Сервер, разбор HTTP и документацию даёт фреймворк; наше предметное действие остаётся функцией Python.

In [ ]:
try:
    from fastapi import FastAPI
except ModuleNotFoundError:
    print("Установите зависимости из lesson04/examples/requirements.txt")
else:
    app = FastAPI()

    @app.get("/hello")
    def hello(name: str = "world") -> dict[str, str]:
        return {"message": f"Hello, {name}!"}

    print(hello("Student"))

## Telegram bot: получить JSON, отправить JSON

[`telegram_echo_bot.py`](examples/telegram_echo_bot.py) использует Bot API напрямую, без дополнительного bot-фреймворка. Цикл вызывает `getUpdates`, берёт текст и `chat_id`, затем вызывает `sendMessage`. `offset = update_id + 1` подтверждает обработанное обновление.

```bash
export TELEGRAM_BOT_TOKEN="..."
python lesson04/examples/telegram_echo_bot.py
```

Вся механика echo-bot — HTTP-запрос, цикл и два условия. Библиотеки для ботов сокращают шаблонный код, но не меняют эту модель. Токен читается из окружения и никогда не коммитится.

Сердце работающего бота выглядит так:

```python
while True:
    updates = requests.get(
        f"{base_url}/getUpdates",
        params={"timeout": 30, "offset": offset},
        timeout=35,
    ).json()["result"]

    for update in updates:
        offset = update["update_id"] + 1
        message = update.get("message")
        if message is None or "text" not in message:
            continue
        requests.post(
            f"{base_url}/sendMessage",
            json={
                "chat_id": message["chat"]["id"],
                "text": f"Вы написали: {message['text']}",
            },
            timeout=10,
        ).raise_for_status()
```

Подготовка токена, проверка HTTP-ошибок и `if __name__ == "__main__"` находятся в полном файле примера.

## Pygame: цикл, события, обновление, рисунок

[`pygame_ball.py`](examples/pygame_ball.py) открывает окно и двигает круг между краями экрана. Внутри обычный `while`:

1. прочитать события и заметить закрытие окна;
2. изменить координату по прошедшему времени;
3. поменять направление у границы;
4. очистить экран, нарисовать круг и показать кадр.

```bash
python lesson04/examples/pygame_ball.py
```

Это минимальный игровой цикл. Игра растёт добавлением состояния и правил, а не появлением отдельного «игрового» языка. Симуляция использует тот же цикл обновления, только обычно без окна.

In [ ]:
try:
    import pygame
except ModuleNotFoundError:
    print("Установите зависимости из lesson04/examples/requirements.txt")
else:
    def run_pygame() -> None:
        pygame.init()
        screen = pygame.display.set_mode((640, 360))
        clock = pygame.time.Clock()
        x = 24.0
        speed = 220.0
        running = True

        while running:
            seconds = clock.tick(60) / 1000
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            x += speed * seconds
            if x >= 616:
                x = 616
                speed = -abs(speed)
            elif x <= 24:
                x = 24
                speed = abs(speed)

            screen.fill("midnightblue")
            pygame.draw.circle(screen, "gold", (x, 180), 24)
            pygame.display.flip()

        pygame.quit()

    print("Запуск: python lesson04/examples/pygame_ball.py")

## Приложения с ИИ

Готовую модель можно использовать как внешний компонент: классифицировать текст, извлекать структуру, искать по документам, предлагать черновик или управлять инструментами. Приложение всё равно должно определить контракт вокруг модели:

- какие данные разрешено отправлять;
- какой формат ответа допустим;
- как проверяется результат;
- что делать при тайм-ауте или неверном ответе;
- где требуется подтверждение человека.

Генеративная модель выдаёт вероятностный результат и может уверенно ошибаться. Ей нельзя без проверки передавать решения с финансовыми, юридическими или иными существенными последствиями.

Реальное ИИ-приложение технически начинается так же, как Telegram bot: обычный HTTP-запрос с токеном из переменной окружения. Отличаются адрес, тело запроса и проверка ответа. Сам вызов занимает несколько строк; основная работа проекта — подготовить данные, определить допустимый результат и решить, что делать при ошибке модели.

На этой лекции намеренно не привязываемся к одному провайдеру. Когда группа выберет конкретный сервис, его минимальный вызов добавляется как внешний адаптер, а не внутрь предметного ядра.

## Почему выбирают Python

- короткий путь от идеи до работающего прототипа;
- читаемый код и большая стандартная библиотека;
- развитая экосистема анализа данных, веба, автоматизации и ML;
- удобное связывание разных систем;
- возможность перенести тяжёлые вычисления в библиотеки, написанные на C, C++, Rust или использующие ускорители.

Скорость разработки и скорость выполнения — разные характеристики. Python часто выигрывает первую и достигает второй за счёт специализированных библиотек.

## Границы применимости

Python не всегда является лучшим выбором:

- жёсткое реальное время и микросекундные гарантии;
- крайне ограниченная память или микроконтроллер без подходящей реализации;
- тяжёлые циклы, которые нельзя передать оптимизированной библиотеке;
- нативный мобильный или сложный браузерный интерфейс;
- поставка одного минимального бинарного файла без среды выполнения;
- система, для которой у команды нет компетенций сопровождения Python.

В реальном продукте языки часто комбинируют. Правильный вопрос — не «можно ли на Python», а «какова полная цена разработки, запуска и поддержки».

## Как будет устроена проектная работа

Проект — это ограниченная учебная работа с четырьмя признаками:

- есть **конкретный результат**, который можно запустить и показать;
- есть **конечный план работ**, составленный самой командой;
- есть **ограниченные рамки курса**, поэтому объём приходится выбирать;
- есть **заинтересованные стороны**: пользователи, команда, куратор и преподаватель.

Процесс идёт от проблемы к работающей демонстрации, а не от длинного списка библиотек.

<img src="assets/project-workflow.svg" alt="Схема проектной работы: от идеи до защиты" style="width: 100%; max-width: 1200px;">

## Базовые правила проекта

- В команде **от 2 до 4 человек**.
- Проект должен быть **работоспособным**: основной сценарий запускается на защите.
- Каждый участник должен понимать, как устроен весь проект, даже если ответственность разделена по модулям.
- Результат должен демонстрировать осмысленное применение Python, библиотек или изученных конструкций.
- Уникальна не обязательно технология сама по себе, а комбинация **предметной задачи и технического решения**. Несколько команд могут использовать FastAPI, но не должны делать один и тот же проект.
- Тема должна быть достаточно интересна команде, чтобы довести её до работающего состояния.

Количество библиотек не является мерой качества. Маленькое законченное приложение лучше набора не связанных между собой интеграций.

## Что регистрирует команда

Конкретная задача выбирается из предложенного списка или отдельно согласуется с преподавателем. Организационный канал регистрации будет объявлен отдельно. В записи о проекте должны быть:

| Поле | Что написать |
| --- | --- |
| Состав | имена 2–4 участников |
| Рабочее название | короткое имя, которое можно изменить позднее |
| Предметная область | где возникает задача |
| Пользователь и проблема | кому и какое действие хотим упростить |
| Ожидаемый результат | что именно запустим и покажем |
| Технологии | предварительный, а не окончательный набор |
| Куратор | преподаватель или ассистент, с которым согласуются контрольные точки |

Межгрупповая команда возможна после организационного согласования. Состав, идея и куратор фиксируются до основной разработки, но технические детали можно уточнять по мере появления работающего прототипа.

## Кто за что отвечает

| Роль | Ответственность |
| --- | --- |
| Команда | выбрать объём, составить план, писать и проверять код, вести репозиторий, демонстрировать результат |
| Куратор | обсуждать постановку, помогать сузить объём, давать обратную связь на контрольных точках |
| Преподаватель | согласовать общие правила, требования и формат итоговой защиты |
| Пользователь | дать контекст задачи и, когда возможно, проверить основной сценарий |

Куратор помогает принять решение, но не проектирует и не пишет приложение вместо команды. Если совет куратора непонятен, команда должна переспросить и зафиксировать принятое решение в issue, README или короткой заметке.

## Проект начинается не с фреймворка

Слабая постановка: «сделаем бота с ИИ и базой данных». Она перечисляет технологии, но не объясняет ценность.

Рабочая постановка отвечает на вопросы:

1. кто конкретно пользуется результатом;
2. какое действие сейчас долго, дорого или неудобно;
3. какие данные доступны;
4. что приложение показывает или изменяет;
5. как проверить успех на демонстрации;
6. что не входит в первую версию.

Технологии выбирают после этого — по требованиям задачи.

## Фильтр проектной идеи

Оцените кандидатную идею по пяти вопросам:

| Вопрос | Хороший признак | Риск |
| --- | --- | --- |
| Есть пользователь? | можно назвать роль и действие | «полезно всем» |
| Есть данные? | формат и источник понятны | данные обещаны когда-нибудь |
| Есть предметное ядро? | правило или алгоритм можно проверить отдельно | вся работа — внешний API |
| Есть демонстрация? | виден вход, действие и результат | успех нельзя наблюдать |
| Помещается в курс? | минимальная версия мала | обязательны десять интеграций |

Лучше маленькое законченное ядро с ясным расширением, чем большой список технологий без работающего сценария.

## Минимальная демонстрируемая версия

MVP курса — не «плохая версия продукта», а минимальный вертикальный сценарий, который проходит через всё приложение:

`реальный вход → проверка → предметное действие → наблюдаемый результат`.

Сначала достаточно одного типа пользователя, одного сценария и небольшого набора данных. После работающего вертикального среза можно добавлять интерфейсы, хранение, конкурентность и поставку.

## Этапы без привязки к датам

Календарные сроки публикуются отдельно. В лекции фиксируем последовательность и результат каждого этапа:

| Этап | Наблюдаемый результат |
| --- | --- |
| Идея и регистрация | состав команды, пользователь, проблема, границы первой версии и куратор |
| Предметное ядро | функции или классы с проверяемыми правилами, первый сквозной сценарий |
| Архитектура и качество | модули, аннотации, тесты, конфигурация и обработка ошибок |
| Алгоритмическая часть | обоснованный алгоритм или структура данных и оценка сложности |
| Интерфейсы и хранение | только действительно нужные API, бот, UI, файлы или база данных |
| Поставка и защита | воспроизводимый запуск, README, демонстрация и ответы команды |

Этап не требует «закончить весь проект». Он требует показать конкретный артефакт, получить обратную связь и обновить следующий короткий план.

## Первая контрольная точка

После занятия группа готовит:

- выбранную предметную область и краткую постановку задачи;
- пользователя, вход, выход и критерий успеха;
- границы первой версии;
- состав команды и первоначальное распределение ответственности;
- репозиторий с README;
- короткий план ближайших работ.

Фреймворк, база данных и облачный деплой пока не обязательны. Решения можно пересматривать, когда требования становятся яснее.

## Как устроена помощь

На занятиях разбираются общие принципы и базовые технологии курса. Куратор помогает с постановкой, декомпозицией, архитектурой и конкретными препятствиями. Для полезного вопроса команда приносит:

1. ожидаемое поведение;
2. минимальный воспроизводимый пример;
3. фактический результат или текст ошибки;
4. уже проверенные гипотезы.

Экзотическую библиотеку или сервис можно выбрать после согласования, но пошаговая экспертиза по любому внешнему инструменту не гарантируется. Риск такой технологии входит в план команды: должен существовать упрощённый путь к работающей демонстрации.

## Когда проект можно считать готовым

Перед защитой команда проверяет не число реализованных функций, а готовность результата:

- основной пользовательский сценарий работает от входа до результата;
- проект запускается по инструкции в чистом окружении;
- предметное ядро отделено от внешних интерфейсов;
- существенные правила покрыты автоматическими тестами;
- ошибки внешних систем обрабатываются явно;
- README объясняет задачу, архитектуру, установку и запуск;
- секреты и персональные данные не лежат в репозитории;
- каждый участник может объяснить архитектуру и свой вклад.

Если без автора проект невозможно запустить, он ещё не готов, даже если однажды работал на его ноутбуке.

## Что показывает команда на защите

Хорошая защита строится вокруг одного связного сценария:

1. назвать пользователя и проблему;
2. запустить проект по записанной инструкции;
3. показать основной вход, действие и результат;
4. объяснить архитектуру без перечисления каждого файла;
5. разобрать одно содержательное техническое решение и его альтернативы;
6. честно назвать ограничения и следующий возможный шаг;
7. ответить на вопросы о коде и вкладе участников.

Записанное видео можно иметь как резерв на случай внешнего сбоя, но оно не заменяет работоспособный проект и понимание кода.

## Как проект связан с оценкой

Проект — один из четырёх компонентов итоговой оценки курса наряду с семинарами, домашними работами и экзаменом. Точные коэффициенты будут объявлены отдельно.

При разборе проекта важны:

- соответствие результата заявленной задаче;
- работоспособность и воспроизводимость;
- качество предметного ядра, тестов и обработки ошибок;
- обоснованность технологий и алгоритмов;
- понимание проекта участниками;
- ясность демонстрации и документации.

Само по себе количество фреймворков, API или строк кода дополнительных баллов не создаёт.

## Официальные руководства

Минимальные примеры можно расширять по первичным источникам:

- [FastAPI: First Steps](https://fastapi.tiangolo.com/tutorial/first-steps/);
- [Telegram Bot API](https://core.telegram.org/bots/api) и [FAQ о получении updates](https://core.telegram.org/bots/faq);
- [Pygame: Quick start](https://www.pygame.org/docs/);
- [scikit-learn: LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

## Неожиданно, но по правилам

Перед запуском предскажите результат и объясните, почему такое поведение важно для прикладной программы.

1. Псевдослучайный генератор с одинаковым seed выдаёт одинаковую последовательность. Это полезно для воспроизводимой симуляции, но не подходит для секретных токенов.
2. `Path.suffix` возвращает только последнее расширение. Для `report.csv.gz` это `.gz`; все расширения доступны через `suffixes`.
3. Стандартный `json.dumps` разрешает `NaN`, хотя строгий JSON такого числа не содержит. Параметр `allow_nan=False` превращает скрытую несовместимость в явную ошибку.

In [ ]:
from pathlib import Path
from random import Random
import json

first = Random(42)
second = Random(42)
print([first.randint(1, 10) for _ in range(4)])
print([second.randint(1, 10) for _ in range(4)])

archive = Path("report.csv.gz")
print(archive.suffix, archive.suffixes)

print(json.dumps({"value": float("nan")}))
try:
    json.dumps({"value": float("nan")}, allow_nan=False)
except ValueError as error:
    print(type(error).__name__, error)

## Самопроверка

1. Где в `csv_report.py` находятся вход, вычисление и выход?
2. Почему массовое переименование сначала стоит проверить во временном каталоге?
3. Что делают `fit` и `predict` в примере scikit-learn?
4. Что добавляет `@app.get("/hello")` к обычной функции Python?
5. Зачем Telegram bot хранит `offset = update_id + 1`?
6. Какие четыре действия повторяет игровой цикл Pygame?
7. Что приложение должно проверять вокруг генеративной модели?
8. В каких задачах Python может быть неудачным выбором?
9. Почему список технологий не является постановкой проекта?
10. Что должно пройти через минимальный вертикальный сценарий?
11. Что команда должна указать при регистрации проекта?
12. Чем ответственность куратора отличается от ответственности команды?
13. Какой наблюдаемый результат требуется на каждой контрольной точке?
14. Какие признаки показывают, что проект действительно готов к защите?

## Итоги

- CLI-отчёт — это чтение аргументов, файла, цикл и печать результата.
- FastAPI endpoint — обычная функция, связанная декоратором с HTTP-маршрутом.
- Echo-bot — цикл из `getUpdates` и `sendMessage`.
- Pygame — цикл обработки событий, изменения состояния и отрисовки.
- `fit` и `predict` запускают реальную ML-модель, но качество определяют данные и проверка.
- Фреймворки снимают инфраструктурный шаблонный код, а предметные правила остаются Python-функциями.
- Проект выполняет команда из 2–4 человек: она регистрирует задачу и состав, сама составляет план и показывает работающий результат.
- Куратор даёт обратную связь, а ответственность за решения, код и воспроизводимый запуск остаётся у команды.
- Проект начинается с пользователя, проблемы и критерия успеха, а не со списка библиотек.

На [семинаре](seminar.ipynb) превратим предметную функцию в устанавливаемый пакет с CLI, проверками и воспроизводимой Git-историей.